<img src="https://wamugii.kento.rw/logo-emblem-128.png" width="56" align="right" alt="WAMUGII">

**Modern AI: LLMs & Generative AI** · Week 1 · How Large Language Models Work · Lesson 2 of 24

# Tokens: How LLMs Read Text

**How to use this notebook:** run the code cells **in order**, from top to bottom (press ▶ on each cell, or use **Runtime → Run all**). You can change any code and run it again. To keep your changes, use **File → Save a copy in Drive**.

The lesson, quiz and certificate are on [WAMUGII](https://wamugii.kento.rw/courses/modern-ai-llms-generative).

**Learning objective:** Explain what tokens are, build a byte-pair-encoding (BPE) tokenizer from scratch, and explain why token counts matter for cost, context limits and fairness across languages.

## LLMs don't see words — they see tokens

Before an LLM reads anything, a **tokenizer** cuts the text into **tokens**: common words become one token, rarer words are split into pieces. "unbelievable" might become "un", "believ", "able". Each token has an id number, and the model only ever sees those numbers.

Why not whole words? There are too many (every name, typo and language). Why not single letters? Sequences become very long and slow. **Sub-word tokens** are the compromise.

Rough rule of thumb for English: **1 token ≈ ¾ of a word**, or about 4 characters.

## Byte-pair encoding (BPE)

Most LLM tokenizers use BPE:

1. Start with single characters.
2. Find the **most frequent pair** of neighbouring tokens in the training text.
3. **Merge** that pair into a new token. Repeat thousands of times.

Frequent words end up as single tokens; rare words stay split.

In [ ]:
import random

random.seed(0)
names = ["Aline", "Eric", "Grace", "Jean", "Diane", "Patrick", "Claudine", "Samuel", "Ange", "Didier"]
places = ["Kigali", "Huye", "Musanze", "Rubavu", "Nyagatare", "Rusizi", "Muhanga"]
goods = ["tomatoes", "beans", "bananas", "avocados", "maize", "potatoes", "milk", "coffee", "rice", "cassava"]
verbs = ["sells", "buys", "carries", "cooks", "packs", "counts"]
times = ["in the morning", "at noon", "in the evening", "on Saturday", "after the rain", "before school"]
moods = ["happy", "tired", "busy", "proud", "calm", "hungry"]
templates = [
    "{n} {v} {g} in {p} {t}.",
    "In {p}, {n} {v} {g} {t}.",
    "{n} is {f} because the {g} are fresh.",
    "{t}, {n} goes to the market in {p}.",
    "{n} pays for the {g} with mobile money.",
    "The price of {g} in {p} is low {t}.",
    "{n} tells {m} that the {g} from {p} are the best.",
    "{n} and {m} share {g} {t}.",
]

def sentence():
    n, m = random.sample(names, 2)
    s = random.choice(templates).format(n=n, m=m, v=random.choice(verbs), g=random.choice(goods),
                                        p=random.choice(places), t=random.choice(times), f=random.choice(moods))
    return s[0].upper() + s[1:]

text = "\n".join(sentence() for _ in range(4000))
print(len(text), "characters")
print(text[:250])

## Build BPE from scratch

In [ ]:
from collections import Counter

def get_pairs(tokens):
    return Counter(zip(tokens, tokens[1:]))

def merge(tokens, pair, new_token):
    out, i = [], 0
    while i < len(tokens):
        if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
            out.append(new_token)
            i += 2
        else:
            out.append(tokens[i])
            i += 1
    return out

def train_bpe(training_text, n_merges):
    tokens = list(training_text)
    merges = []
    for _ in range(n_merges):
        pairs = get_pairs(tokens)
        if not pairs:
            break
        best = pairs.most_common(1)[0][0]
        merges.append(best)
        tokens = merge(tokens, best, best[0] + best[1])
    return merges

merges = train_bpe(text[:20000], n_merges=150)
print("first merges:", merges[:8])
print("later merges:", merges[-5:])

The first merges join very common pairs such as "e" + " " and "s" + " "; later merges build whole words and phrases such as "the ", "Eric" and even "after the rain".

## Encode new text

In [ ]:
def encode_bpe(s, merges):
    tokens = list(s)
    for pair in merges:                      # apply merges in the order they were learned
        tokens = merge(tokens, pair, pair[0] + pair[1])
    return tokens

sentence = "Grace sells avocados in Kigali in the morning."
tokens = encode_bpe(sentence, merges)
print(tokens)
print(len(sentence), "characters ->", len(tokens), "tokens")

The 46-character sentence becomes about 21 tokens: frequent pieces have become single tokens, while the rarer name "Kigali" is still split into small parts.

## The language gap

Our tokenizer learned only from English market sentences. Watch what happens with text it has not seen much of:

In [ ]:
samples = {
    "English": "Grace pays for the beans with mobile money.",
    "Kinyarwanda": "Grace yishyura ibishyimbo akoresheje mobile money.",
    "Made-up word": "Xqzvbrtwplk",
}
for language, s in samples.items():
    t = encode_bpe(s, merges)
    print(f"{language:12s} {len(s):3d} characters -> {len(t):3d} tokens   ({len(s) / len(t):.1f} characters per token)")

The English sentence becomes only **8** tokens (more than 5 characters each). The same meaning in Kinyarwanda needs about **36** tokens — the tokenizer barely knows the language, so it falls back to tiny pieces — and the random word is cut into single letters. Real tokenizers show the same pattern: text in languages that were rare in the tokenizer's training data — including many African languages — needs **more tokens** for the same meaning. That means:

- **Higher cost** — APIs charge per token.
- **Less fits in the context window** — the model "sees" less of your document.
- Often **lower quality**, because the model saw less of the language.

Newer models are improving their coverage of African languages, and projects across Africa are building better data for them.

## Tokens decide cost and limits

- **Pricing:** LLM APIs charge per million input tokens and per million output tokens (output usually costs more).
- **Context window:** the maximum number of tokens the model can consider at once — from a few thousand in early models to hundreds of thousands or more today. Your prompt, documents, conversation history **and** the answer must all fit.
- **Spelling and counting:** because the model sees tokens, not letters, tasks like "how many r's are in strawberry" or reversing a word can trip it up.

## Key terms

- **Token** — a piece of text the model reads as one unit, with an id number.
- **Tokenizer** — the tool that splits text into tokens and back.
- **BPE** — byte-pair encoding: repeatedly merge the most frequent pair.
- **Vocabulary** — all the tokens a tokenizer knows (often 50,000–200,000).
- **Context window** — the most tokens a model can take in at once.

**Takeaway:** LLMs read sub-word tokens; how a language is tokenized affects price, how much text fits, and quality — something to watch when building for Kinyarwanda and other African languages.

## Try it yourself

1. Train BPE with 50, 150 and 400 merges. How does the token count of the English sentence change?
2. Add 20 Kinyarwanda sentences to the training text and retrain. Does the Kinyarwanda token count drop?
3. Write a `decode_bpe` function that joins tokens back into text. Is it always perfect?
4. Paste the same paragraph in English and in Kinyarwanda into an online LLM tokenizer tool and compare the counts.

---
Finished? Go back to [Modern AI: LLMs & Generative AI on WAMUGII](https://wamugii.kento.rw/courses/modern-ai-llms-generative), tick the lesson as done and take the quiz.

*© WAMUGII TECH SOLUTIONS · Original course material for WAMUGII AI Program students.*